# Notebook: 01_exploracao_corpus_jornalistico.ipynb
**id_rastreabilidade:** RTB_001  
**Versão:** v02  
**Data de criação:** "24/03/2026"  
**Última atualização:** "09/05/2026"  
**Responsável:** Ailton Vendramini  

---

## Finalidade
Explorar e consolidar o corpus jornalístico do Atlas Social de Hortolândia.  
Todos os CSVs estão no schema **v10.4** (20 colunas) após migração completa em mai/2026.

---

## Schema v10.4 — 20 colunas
`id_evento` · `data_publicacao` · `fonte` · `titulo` · `pagina` · `municipio` · `localidade`  
`tipo_camada` · `dimensao_analitica` · `tipo_relacao_variavel` · `codigo_variavel`  
`nivel_criticidade` · `observacao` · `aproximacao_variavel` · `nota_analista`  
`cod_loteamento` · `nivel_confianca_loteamento` · `papel_no_ciclo` · `id_caso_pressao` · `entra_ipst`

---

## Outputs Gerados
| Caminho | tipo_output | Commitar? |
|---|---|---|
| `outputs/tabelas/corpus_consolidado.csv` | exploratorio | Não |

---

## Changelog
- v02 (09/05/2026): migração completa para schema v10.4; remoção de células legacy
- v01 (24/03/2026): exploração inicial (schema antigo)


In [ ]:
import pandas as pd
import glob
import os

# ── Configuração ──────────────────────────────────────────────────────────────
PASTA = r'C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas'

COLUNAS_V10_4 = [
    'id_evento', 'data_publicacao', 'fonte', 'titulo', 'pagina', 'municipio',
    'localidade', 'tipo_camada', 'dimensao_analitica', 'tipo_relacao_variavel',
    'codigo_variavel', 'nivel_criticidade', 'observacao', 'aproximacao_variavel',
    'nota_analista', 'cod_loteamento', 'nivel_confianca_loteamento',
    'papel_no_ciclo', 'id_caso_pressao', 'entra_ipst'
]

print(f"Pasta: {PASTA}")
print(f"Schema v10.4: {len(COLUNAS_V10_4)} colunas")

In [ ]:
# ── Carregar todos os CSVs ────────────────────────────────────────────────────
arquivos = sorted(glob.glob(os.path.join(PASTA, "2026_*.csv")))
print(f"Arquivos encontrados: {len(arquivos)}")

dfs = []
ignorados = []

for arq in arquivos:
    try:
        df_tmp = pd.read_csv(arq, sep=",", encoding="utf-8", dtype=str)
        # verifica se tem id_evento (schema v10.4)
        if 'id_evento' not in df_tmp.columns:
            ignorados.append(os.path.basename(arq))
            continue
        # remove coluna peso residual se existir
        df_tmp = df_tmp.drop(columns=['peso'], errors='ignore')
        dfs.append(df_tmp)
    except Exception as e:
        print(f"ERRO em {os.path.basename(arq)}: {e}")

if ignorados:
    print(f"\nIgnorados (schema incompativel): {len(ignorados)}")
    for i in ignorados:
        print(f"  - {i}")

if not dfs:
    raise ValueError("Nenhum arquivo carregado. Verifique o caminho e os schemas.")

df_total = pd.concat(dfs, ignore_index=True, sort=False)

# remove evento politico Clodoaldo (fora do escopo SMIDS)
df_total = df_total[
    ~df_total['titulo'].str.contains("Clodoaldo", na=False)
]

print(f"\n=== BASE CONSOLIDADA ===")
print(f"Total de eventos: {len(df_total)}")
print(f"Colunas: {list(df_total.columns)}")

In [ ]:
# ── Verificação de integridade ────────────────────────────────────────────────
print("Colunas presentes:")
print(df_total.columns.tolist())

print(f"\nPeriodo coberto:")
print(f"  Início : {df_total['data_publicacao'].min()}")
print(f"  Fim    : {df_total['data_publicacao'].max()}")
print(f"  Edições: {df_total['data_publicacao'].nunique()}")

print(f"\nAmostra (5 registros):")
df_total[['id_evento','data_publicacao','titulo','tipo_camada','nivel_criticidade']].head(5)

In [ ]:
# ── Eventos por data de publicação ───────────────────────────────────────────
print("Eventos por data (top 20):")
df_total['data_publicacao'].value_counts().sort_index().tail(20)

In [ ]:
# ── Distribuição por tipo_camada ──────────────────────────────────────────────
print("Distribuição por tipo_camada:")
df_total['tipo_camada'].value_counts()

In [ ]:
# ── Distribuição por dimensao_analitica ───────────────────────────────────────
print("Dimensões analíticas:")
df_total['dimensao_analitica'].value_counts()

In [ ]:
# ── Distribuição por nivel_criticidade ────────────────────────────────────────
print("Nível de criticidade:")
df_total['nivel_criticidade'].value_counts()

In [ ]:
# ── Distribuição por tipo_relacao_variavel ────────────────────────────────────
print("Tipo de relação com variável IVS:")
df_total['tipo_relacao_variavel'].value_counts()

In [ ]:
# ── Distribuição por papel_no_ciclo ───────────────────────────────────────────
print("Papel no ciclo de pressão:")
df_total['papel_no_ciclo'].value_counts()

In [ ]:
# ── Ciclos de pressão ativos ──────────────────────────────────────────────────
ciclos = (
    df_total[df_total['id_caso_pressao'].notna() & (df_total['id_caso_pressao'] != '')]
    .groupby('id_caso_pressao')
    .agg(
        n_registros=('id_evento', 'count'),
        ultimo_evento=('data_publicacao', 'max'),
        ultimo_papel=('papel_no_ciclo', 'last')
    )
    .sort_values('ultimo_evento', ascending=False)
)
print("Ciclos identificados no corpus:")
ciclos

In [ ]:
# ── Distribuição por município ────────────────────────────────────────────────
print("Distribuição por município:")
df_total['municipio'].value_counts()

In [ ]:
# ── Eventos que entram no IPST-H ─────────────────────────────────────────────
print("Eventos que alimentam o IPST-H (entra_ipst = sim):")
ipst = df_total[df_total['entra_ipst'].str.lower() == 'sim']
print(f"Total: {len(ipst)}")
ipst[['data_publicacao','titulo','tipo_camada','dimensao_analitica','papel_no_ciclo','id_caso_pressao']]

In [ ]:
# ── Eventos IVS com relação direta (dado auditável) ──────────────────────────
ivs_direto = df_total[
    (df_total['tipo_camada'] == 'IVS') &
    (df_total['tipo_relacao_variavel'] == 'direta')
][['data_publicacao','titulo','codigo_variavel','nivel_criticidade','nota_analista']]

print(f"Eventos IVS / relação direta: {len(ivs_direto)}")
ivs_direto

In [ ]:
# ── PRESSAO_SOCIAL com nivel_criticidade = critico ou alerta ─────────────────
pressao_alta = df_total[
    (df_total['tipo_camada'] == 'PRESSAO_SOCIAL') &
    (df_total['nivel_criticidade'].isin(['critico', 'alerta']))
][['data_publicacao','municipio','localidade','titulo','papel_no_ciclo','id_caso_pressao']]

print(f"Eventos de alta pressão: {len(pressao_alta)}")
pressao_alta.sort_values('data_publicacao', ascending=False).head(20)

In [ ]:
# ── Exportar base consolidada ─────────────────────────────────────────────────
output_path = r'C:\Users\ailto\Atlas-Social-de-Hortolandia\outputs\tabelas\corpus_consolidado.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_total.to_csv(output_path, index=False, encoding='utf-8')
print(f"✅ Exportado: {output_path}")
print(f"   Eventos: {len(df_total)} | Colunas: {len(df_total.columns)}")